In [ ]:
import requests
import pandas as pd
import numpy as np
from tqdm import tqdm
from datetime import timedelta, datetime
from math import radians, sin, cos, sqrt, atan2

In [ ]:
ports_dict = {
    "Halifax": {"lat": 44.648, "lon": -63.57},
    "Gdansk": {"lat": 54.37815, "lon": 18.67575},
    "Gothenburg": {"lat": 57.6995, "lon": 11.883},
    "Antwerp": {"lat": 51.30249, "lon": 4.31146},
    "Brugge": {"lat": 51.22984, "lon": 3.224471},
    "Southampton": {"lat": 50.89817, "lon": -1.420503},
    "Le Havre": {"lat": 49.47265, "lon": 0.1462042},
    "Santander": {"lat": 43.45535, "lon": -3.808037},
    "Vigo": {"lat": 42.2415, "lon": -8.721235},
    "Emden": {"lat": 53.3475, "lon": 7.191},
    "Bremerhaven": {"lat": 53.55, "lon": 8.58},
    "Galveston": {"lat": 29.31, "lon": -94.793},
    "Brunswick": {"lat": 31.145, "lon": -81.493},
    "Savannah": {"lat": 32.104, "lon": -81.151},
    "Charleston": {"lat": 32.79, "lon": -79.925},
    "Wilmington": {"lat": 39.736, "lon": -75.515},
    "New York - Port Newark": {"lat": 40.684, "lon": -74.155},
    "Davisville": {"lat": 41.588, "lon": -71.404},
}

moving_status = [0, 3, 4, 8, 11, 12]
staying_status = [1, 2, 5, 6, 7, 9, 10, 13]

def get_port_by_coords(lat, lon):
    lat_int = int(lat)
    lon_int = int(lon)
    for port, info in ports_dict.items():
        if int(info["lat"]) == lat_int and int(info["lon"]) == lon_int:
            return port
    return None

In [1]:
weather_features = [
    "wave_height", "wave_direction", "wave_period",
    "wind_wave_height", "wind_wave_direction", "wind_wave_period",
    "swell_wave_height", "swell_wave_direction", "swell_wave_period",
    "ocean_current_velocity", "ocean_current_direction"
]

def fetch_weather(lat, lon, date, time):
    url = (
        f"https://customer-marine-api.open-meteo.com/v1/marine?latitude={lat}&longitude={lon}"
        f"&hourly={','.join(weather_features)}"
        f"&start_date={date}&end_date={date}"
        f"&apikey=FAambAt8nVfSOqCM"
    )
    try:
        resp = requests.get(url)
        resp.raise_for_status()
        data = resp.json()
        # Find the closest hour to the given time
        times = data["hourly"]["time"]
        # Find index of closest time
        time_diffs = [abs(pd.Timestamp(t) - pd.Timestamp(time)) for t in times]
        idx = time_diffs.index(min(time_diffs))
        # Extract weather data for that hour
        weather = {feat: data["hourly"][feat][idx] for feat in weather_features}
        return weather
    except Exception as e:
        print(f"Weather API error for {lat},{lon} {date}: {e}")
        return {feat: None for feat in weather_features}    

In [ ]:
wind_features = ["wind_speed_10m", "wind_direction_10m"]

def fetch_wind(lat, lon, date, time):
    url = (
        f"https://customer-archive-api.open-meteo.com/v1/archive?latitude={lat}&longitude={lon}"
        f"&hourly={','.join(wind_features)}"
        f"&start_date={date}&end_date={date}"
        f"&apikey=FAambAt8nVfSOqCM"
    )
    try:
        resp = requests.get(url)
        resp.raise_for_status()
        data = resp.json()
        times = data["hourly"]["time"]
        # Find index of closest time
        time_diffs = [abs(pd.Timestamp(t) - pd.Timestamp(time)) for t in times]
        idx = time_diffs.index(min(time_diffs))
        wind = {feat: data["hourly"][feat][idx] for feat in wind_features}
        return wind
    except Exception as e:
        print(f"Wind API error for {lat},{lon} {date}: {e}")
        return {feat: None for feat in wind_features}

In [ ]:
weather_features_cities = [
    "temperature_2m", "precipitation_probability", "precipitation", "weathercode",
    "wind_speed_10m", "wind_gusts_10m", "wind_direction_10m", "visibility",
    "pressure_msl", "snowfall", "snow_depth"
]

def fetch_city_weather(lat, lon, date, time):
    url = (
        f"https://customer-archive-api.open-meteo.com/v1/archive?latitude={lat}&longitude={lon}"
        f"&hourly={','.join(weather_features_cities)}"
        f"&start_date={date}&end_date={date}"
        f"&apikey=FAambAt8nVfSOqCM"
    )
    try:
        resp = requests.get(url)
        resp.raise_for_status()
        data = resp.json()
        times = data["hourly"]["time"]
        # Find index of closest time
        time_diffs = [abs(pd.Timestamp(t) - pd.Timestamp(time)) for t in times]
        idx = time_diffs.index(min(time_diffs))
        weather = {feat: data["hourly"][feat][idx] for feat in weather_features_cities}
        return weather
    except Exception as e:
        print(f"Weather API error for {lat},{lon} {date}: {e}")
        return {feat: None for feat in weather_features_cities}

In [ ]:
all_journeys = pd.read_csv('all_journeys.csv')
all_journeys['date'] = pd.to_datetime(all_journeys['date'])
all_journeys = all_journeys.sort_values(['journey_id', 'mmsi', 'date']).reset_index(drop=True)
all_journeys.to_pickle('all_journeys.pkl')

In [ ]:
# Add destinations
weather_df = pd.read_pickle("all_journeys.pkl")

weather_df = weather_df.sort_values(['journey_id', 'date']).reset_index(drop=True)
weather_df['destination'] = None
weather_df['nav_status'] = weather_df['nav_status'].astype(int)

for journey_id, group in weather_df.groupby('journey_id'):
    idx = group.index
    i = 0
    while i < len(idx):
        row = weather_df.loc[idx[i]]
        # If moving, look for next staying + port
        if row['nav_status'] in moving_status:
            start = i
            found_port = None
            end = None
            for j in range(i+1, len(idx)):
                next_row = weather_df.loc[idx[j]]
                if next_row['nav_status'] in staying_status:
                    port = get_port_by_coords(next_row['lat'], next_row['lon'])
                    if port:
                        found_port = port
                        end = j
                        break
            if found_port and end is not None:
                # Mark all from start to end (inclusive) with port as destination
                weather_df.loc[idx[start:end+1], 'destination'] = found_port
                # Mark all staying-in-port entries after the first with None
                k = end
                # Find all consecutive staying-in-port entries at this port
                while k < len(idx):
                    curr_row = weather_df.loc[idx[k]]
                    curr_port = get_port_by_coords(curr_row['lat'], curr_row['lon'])
                    if curr_row['nav_status'] in staying_status and curr_port == found_port:
                        if k == end:
                            # First staying-in-port entry: keep destination
                            pass
                        else:
                            weather_df.loc[idx[k], 'destination'] = None
                        k += 1
                    else:
                        break
                i = k
            else:
                i += 1
        else:
            i += 1

# Add destination lat/lon columns by matching unlocodes
weather_df['destination_lat'] = weather_df['destination'].map(lambda x: ports_dict[x]['lat'] if x in ports_dict else None)
weather_df['destination_lon'] = weather_df['destination'].map(lambda x: ports_dict[x]['lon'] if x in ports_dict else None)

In [ ]:
# Add marine weather at coordinates X Y
weather_data = []
for _, row in tqdm(weather_df.iterrows(), total=len(weather_df), desc="Fetching weather"):
    lat = row["lat"]
    lon = row["lon"]
    timestamp = row["date"]
    date = str(timestamp)[:10]  # YYYY-MM-DD
    weather = fetch_weather(lat, lon, date, timestamp)
    weather_data.append(weather)

weather_df = pd.concat([weather_df.reset_index(drop=True), pd.DataFrame(weather_data)], axis=1)

In [ ]:
# Adding wind on sea
wind_data = []
for _, row in tqdm(weather_df.iterrows(), total=len(weather_df), desc="Fetching wind data"):
    lat = row["lat"]
    lon = row["lon"]
    timestamp = row["date"]
    date = str(timestamp)[:10]  # YYYY-MM-DD
    wind = fetch_wind(lat, lon, date, timestamp)
    entry = {
        "journey_id": row["journey_id"],
        "lat": lat,
        "lon": lon,
        "date": timestamp,
        **wind
    }
    wind_data.append(entry)

wind_df = pd.DataFrame(wind_data)
# Merge wind_df columns into weather_df
weather_df = pd.merge(
    weather_df,
    wind_df[["journey_id", "lat", "lon", "date", "wind_speed_10m", "wind_direction_10m"]],
    on=["journey_id", "lat", "lon", "date"],
    how="left"
)

weather_df.to_pickle("full_weather.pkl")
weather_df.to_csv("full_weather.csv", index=False)
weather_df.head()

In [ ]:
# Optionally, add weather at destination ports
"""
weather_df = pd.read_pickle("full_weather.pkl")

city_weather_data = []
for _, row in tqdm(weather_df.iterrows(), total=len(weather_df), desc="Fetching port weather"):
    dest_lat = row["destination_lat"]
    dest_lon = row["destination_lon"]
    timestamp = row["date"]
    date = str(timestamp)[:10]  # YYYY-MM-DD
    if pd.isna(dest_lat) or pd.isna(dest_lon):
        weather = {feat: None for feat in weather_features_cities}
    else:
        weather = fetch_city_weather(dest_lat, dest_lon, date, timestamp)
    entry = {
        "journey_id": row["journey_id"],
        "destination": row["destination"],
        "date": timestamp,
        "destination_lat": dest_lat,
        "destination_lon": dest_lon,
        **weather
    }
    city_weather_data.append(entry)

destination_weather_df = pd.DataFrame(city_weather_data)
destination_weather_df.to_pickle("historical_cities_weather.pkl")
destination_weather_df.to_csv("historical_cities_weather.csv", index=False)
"""